# 00 CIK matcher

Adds SEC Central Index Keys to the S&P 1500 constituent list (`SP1500.csv` -> `SP1500_CIK.csv`, stored as `data/SP1500_CIK.csv`). No thesis table.

In [1]:
import re
import time
import json
from pathlib import Path

import pandas as pd
import requests

In [3]:
# ----------------------------
# Config
# ----------------------------
INPUT_CSV = Path("SP1500.csv")
OUTPUT_CSV = Path("SP1500_CIK.csv")

# SEC CIK mapping file (EDGAR data).
SEC_TICKER_MAP_URL = "https://www.sec.gov/files/company_tickers_exchange.json"

HEADERS = {
    # SEC expects a descriptive User-Agent with contact details
    "User-Agent": "Sören Schlißke (soeren.schlisske@student.unisg.ch)",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}

In [ ]:
# ----------------------------
# Helpers
# ----------------------------
def normalize_ticker(ticker: str) -> str:
    """
    SEC mapping commonly uses '-' for share classes (e.g., BRK-B),
    while many lists use '.' (e.g., BRK.B). Normalize to SEC style.
    """
    if pd.isna(ticker):
        return ""
    t = str(ticker).strip().upper()
    t = t.replace(".", "-")
    t = re.sub(r"\s+", "", t)
    return t

def cik_to_10digits(cik: int | str) -> str:
    """Return CIK as 10-digit, zero-padded string (EDGAR convention)."""
    cik_int = int(cik)
    return f"{cik_int:010d}"

def download_sec_ticker_map(url: str = SEC_TICKER_MAP_URL) -> dict:
    """
    Downloads SEC's company_tickers_exchange.json which includes:
    { "fields": [...], "data": [ [...], ... ] }
    """
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json()

def build_ticker_to_cik_map(sec_json: dict) -> dict[str, str]:
    fields = sec_json["fields"]
    data = sec_json["data"]

    # Identify field indices robustly
    # Typical fields include: cik, name, ticker, exchange
    idx_cik = fields.index("cik")
    idx_ticker = fields.index("ticker")

    mapping = {}
    for row in data:
        ticker = normalize_ticker(row[idx_ticker])
        cik = cik_to_10digits(row[idx_cik])
        # If duplicates exist, keep the first encountered (usually fine)
        mapping.setdefault(ticker, cik)
    return mapping

In [5]:
# ----------------------------
# Main
# ----------------------------
def add_cik_column(sp1500_path: Path = INPUT_CSV, out_path: Path = OUTPUT_CSV) -> pd.DataFrame:
    df = pd.read_csv(sp1500_path)

    # Download SEC ticker map (single request; faster and more polite than per-company calls)
    sec_json = download_sec_ticker_map()
    time.sleep(0.2)  # delay between requests

    ticker_to_cik = build_ticker_to_cik_map(sec_json)

    df["Symbol_norm"] = df["Symbol"].map(normalize_ticker)
    df["CIK"] = df["Symbol_norm"].map(ticker_to_cik)


    # Drop helper column
    df.drop(columns=["Symbol_norm"], inplace=True)

    df.to_csv(out_path, index=False)
    return df


In [6]:
if __name__ == "__main__":
    df_out = add_cik_column()
    missing = df_out["CIK"].isna().sum()
    print(f"Done. Wrote: {OUTPUT_CSV.resolve()}")
    print(f"Rows: {len(df_out)} | Missing CIKs: {missing}")
    if missing:
        print("Tip: common misses are share-class punctuation (e.g., BRK.B vs BRK-B) or non-standard tickers.")

Done. Wrote: C:\Users\Sören Schlißke\Dateien\Uni\MACFin\Thesis\Python\SP1500_CIK.csv
Rows: 1499 | Missing CIKs: 3
Tip: common misses are share-class punctuation (e.g., BRK.B vs BRK-B) or non-standard tickers.
